In [27]:
!pip install pandas-gbq google-auth --quiet

import os
import hashlib
import pandas as pd
import pandas_gbq
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_dark"

PROJECT_ID = "gdelt-494812"
DATASET_ID = "benin_2025"
KEY_PATH = "gdelt-494812-54aad2c4e931.json"


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [28]:
def get_credentials():
    if os.path.exists(KEY_PATH):
        from google.oauth2 import service_account
        return service_account.Credentials.from_service_account_file(KEY_PATH)
    return None

credentials = get_credentials()

In [29]:
os.makedirs("cache/", exist_ok=True)

def charger_donnees(query, force_reload=False):
    query_hash = hashlib.md5(query.encode()).hexdigest()[:8]
    cache_path = f"cache/cache_{query_hash}.csv"
    if os.path.exists(cache_path) and not force_reload:
        df = pd.read_csv(cache_path)
    else:
        df = pandas_gbq.read_gbq(query, project_id=PROJECT_ID, credentials=credentials)
        df.to_csv(cache_path, index=False)
    return df

In [31]:
query_events_head = f"""
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.events_clean`
LIMIT 5
"""

df_events = charger_donnees(query_events_head)
df_events.head()

Downloading: 100%|██████████|


,GLOBALEVENTID,DATEADDED,SQLDATE,MonthYear,Year,Actor1Name,Actor1CountryCode,Actor1Type1Code,Actor2Name,Actor2CountryCode,...,QuadClass_Label,interaction_type,EventCategory,Actor1Role,Actor2Role,goldstein_category,tone_category,has_international_actor,event_scope,is_significant
0,1221339362,20250118064500,20250118,202501,2025,ECOWAS,WAF,IGO,None,None,...,Verbal Cooperation,Cooperation,Make Public Statement,Intergovernmental Org,None,Neutral,Negative,False,Unknown,True
1,1220702354,20250115101500,20250115,202501,2025,ECOWAS,WAF,IGO,BENIN,BEN,...,Verbal Cooperation,Cooperation,Make Public Statement,Intergovernmental Org,Government,Neutral,Very Negative,False,International,True
2,1274392218,20251115230000,20251115,202511,2025,AFRICA,AFR,None,UNITED STATES,USA,...,Verbal Cooperation,Cooperation,Make Public Statement,None,None,Neutral,Negative,True,Unknown,True
3,1248142398,20250606023000,20250606,202506,2025,BENIN,BEN,GOV,None,None,...,Verbal Cooperation,Cooperation,Make Public Statement,Government,None,Neutral,Negative,False,Domestic,True
4,1250982889,20250709024500,20250709,202507,2025,BENIN,BEN,None,None,None,...,Verbal Cooperation,Cooperation,Make Public Statement,None,None,Neutral,Positive,False,Domestic,True


In [32]:
query_gkg_head = f"""
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.gkg_clean`
LIMIT 5
"""

df_gkg = charger_donnees(query_gkg_head)
df_gkg.head()

Downloading: 100%|██████████|


,GKGRECORDID,DATE,SourceCommonName,DocumentIdentifier,V2Themes,V2Locations,V2Persons,V2Organizations,V2Tone,TranslationInfo,...,tone_activity,word_count,tone_category,source_language,is_translated,nb_themes,nb_persons,nb_organizations,nb_locations,is_rich_article
0,20250101003000-152,20250101003000,guardian.ng,https://guardian.ng/news/benin-protests-remark...,"ECON_DEBT,737;WB_1104_MACROECONOMIC_VULNERABIL...",1#Nigeria#NI#NI##10#8#NI#712;1#Benin#BN#BN##9....,"Olushegun Bakari,397;Mohamed Bazoum,921",None,"-5.14018691588785,2.33644859813084,7.476635514...",None,...,17.757009,200.0,Very Negative,eng,False,30,2,0,18,False
1,20250101003000-707,20250101003000,punchng.com,https://punchng.com/five-motorists-escape-deat...,"WOUND,1604;WOUND,2156;CRISISLEX_C03_WELLBEING_...","1#Benin#BN#BN##9.5#2.25#BN#139;5#Ogun State, O...","Abule Oko,2041;Seni Ogunyemi,341","Enforcement Agency,325;Volvo,1380;Volvo,1499","-9.27536231884058,0.869565217391304,10.1449275...",None,...,20.869565,332.0,Very Negative,eng,False,51,2,3,17,True
2,20250101003000-198,20250101003000,punchng.com,https://punchng.com/pdp-seeks-igs-intervention...,"TAX_FNCACT_CHIEF,1138;TAX_FNCACT_CHIEF,2677;TA...","5#Edo State, Edo, Nigeria#NI#NI37#191281#6.5#6...","Justice Daniel Okungbowa,1193;Justice Efe Ikpo...","House Of Assembly,528;House Of Assembly,1542;P...","-11.6822429906542,1.86915887850467,13.55140186...",None,...,18.224299,402.0,Very Negative,eng,False,100,6,4,6,True
3,20250101003000-176,20250101003000,punchng.com,https://punchng.com/fg-votes-n365bn-for-new-ro...,"ENV_SOLAR,3642;WB_841_JUSTICE_SYSTEM_ADMINISTR...","4#Shendam, Nigeria (General), Nigeria#NI#NI00#...","Kwanar Dakatsalle-Falgore,1858;Toyota Hilux,27...","National Assembly,2995;Ministry Of Works,29","-0.599700149925037,1.19940029985008,1.79910044...",None,...,15.442279,625.0,Neutral,eng,False,82,6,2,21,True
4,20250101004500-259,20250101004500,pakobserver.net,https://pakobserver.net/saudi-fund-for-develop...,"WB_2433_CONFLICT_AND_VIOLENCE,126;WB_2432_FRAG...",1#Saudi Arabia#SA#SA##25#45#SA#656;1#Saudi Ara...,None,"Saudi Fund For Development,219","2.9940119760479,4.49101796407186,1.49700598802...",None,...,15.868263,292.0,Positive,eng,False,43,0,1,17,False


In [33]:
query_mentions_head = f"""
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.mentions_clean`
LIMIT 5
"""

df_mentions = charger_donnees(query_mentions_head)
df_mentions.head()

Downloading: 100%|██████████|


,GLOBALEVENTID,MentionTimeDate,MentionType,MentionSourceName,Confidence,MentionDocTone,MentionDocTranslationInfo,mention_datetime,mention_date,mention_year_month,MentionType_Label,tone_category,confidence_level,source_language,is_translated
0,1227182544,20250218050000,1,14ymedio.com,100,1.683502,srclc:spa;eng:GT-SPA 1.0,2025-02-18 05:00:00+00:00,2025-02-18,2025-02,Web,Positive,High,spa,True
1,1227182830,20250218050000,1,14ymedio.com,40,1.683502,srclc:spa;eng:GT-SPA 1.0,2025-02-18 05:00:00+00:00,2025-02-18,2025-02,Web,Positive,Low,spa,True
2,1227182831,20250218050000,1,14ymedio.com,60,1.683502,srclc:spa;eng:GT-SPA 1.0,2025-02-18 05:00:00+00:00,2025-02-18,2025-02,Web,Positive,Medium,spa,True
3,1268569518,20251014160000,1,14ymedio.com,100,3.072626,srclc:spa;eng:GT-SPA 1.0,2025-10-14 16:00:00+00:00,2025-10-14,2025-10,Web,Positive,High,spa,True
4,1268569834,20251014160000,1,14ymedio.com,100,3.072626,srclc:spa;eng:GT-SPA 1.0,2025-10-14 16:00:00+00:00,2025-10-14,2025-10,Web,Positive,High,spa,True


In [30]:
query_lang = f"""
SELECT source_language, tone_category, COUNT(*) as total
FROM `{PROJECT_ID}.{DATASET_ID}.gkg_clean`
WHERE source_language IS NOT NULL
GROUP BY source_language, tone_category
ORDER BY total DESC
LIMIT 30
"""

df_lang = charger_donnees(query_lang)

fig = px.bar(
    df_lang,
    x="source_language",
    y="total",
    color="tone_category",
    barmode="stack",
    title="Sentiment par Langue Source des Articles au Bénin (2025)"
)
fig.update_layout(template="plotly_dark")
fig.show()

## 01 — L'anglais domine la narration d'un pays francophone

**~25 000 articles** en anglais contre **~12 000** en français — soit **2× plus** de couverture anglophone.

Le français béninois produit peu de contenu numérique indexé à l'échelle mondiale.
Le Bénin compte ~100 journaux et ~200 médias audiovisuels *(RSF, 2025)*, mais leur
diffusion reste locale. La francophonie béninoise parle à voix basse sur l'internet
mondial — et GDELT enregistre ce silence.

---

## 02 — Le sentiment négatif est structurel, pas conjoncturel

Dans chaque barre — `eng`, `fra`, `spa`, `por` — le **Négatif** et le **Très Négatif**
occupent la majeure partie de la hauteur, toutes langues confondues.

Ce pattern uniforme transcende les cultures éditoriales. En 2023–2025, le Bénin a connu
la suspension de médias indépendants et des restrictions du Code du numérique
*(Amnesty International, 2025)*. Le sentiment négatif dans les données est le reflet
d'une **presse sous tension**, pas une distorsion algorithmique.

---

## 03 — Neuf langues présentes, mais 50 invisibles

Les codes `eng`, `fra`, `spa`, `ita`, `por`, `rus`, `ell`, `bul`, `zho`, `ara`
représentent la quasi-totalité des articles.

Pourtant, le Bénin compte **plus de 50 langues vivantes** — fon, yoruba, bariba, dendi…
Ces langues, portées par la radio et la tradition orale, ne génèrent pas de texte
numérique indexable. GDELT ne capte pas la réalité vernaculaire du pays.

> **L'absence d'une langue dans les données n'est pas son silence — c'est son exclusion.**